In [0]:
catalog = "vcredit"
bronze_schema_name = "bronze"
silver_schema_name = "silver"
gold_schema_name = "gold"

bronze_schema = f"{catalog}.{bronze_schema_name}"
silver_schema = f"{catalog}.{silver_schema_name}"
gold_schema = f"{catalog}.{gold_schema_name}"

In [0]:
from pyspark.sql import functions as F

df_clientes = spark.table(
    f"{silver_schema}.tb_clientes"
)

dim_cliente = (
    df_clientes

    .withColumn(
        "faixa_etaria",
        F.when(F.col("idade") < 29, "Jovem")
         .when(F.col("idade") < 59, "Adulto")
         .otherwise("Idoso")
    )

    .withColumn(
        "tempo_base_cliente",
        F.round(
            F.months_between(
                F.current_date(),
                F.col("data_cadastro")
            ) / 12,
            1
        )
    )

    .select(
        "id_cliente",
        "nome",
        "email",
        "regiao",
        "idade",
        "faixa_etaria",
        "data_cadastro",
        "tempo_base_cliente",
        "dia_vencimento_fatura"
    )
)

dim_cliente.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(
        f"{gold_schema}.dim_cliente"
    )